# Evaluating the Diagnostic Accuracy of MedGemma-27B: A Comparative Study Against Confirmed Clinical Diagnoses

A study to evaluate the accuracy of the text-to-diagnosis capabilities of MedGemma 1.5 27B when faced with MIMIC-IV patient narrative.

## Introduction

This project will investigate the viability of MedGemma-27B (2025) for clinical text reasoning and probabilistic calibration (diagnosis). MedGemma-27B is a massive, open-weight foundation model specialized for medical text comprehension. In essence, most of their training data is specialized in medical documentation, making it far more suitable for a medical context than traditional, conglomerate-trained LLMs.

## Methodology
A rough chronological methodology developed for the text-to-diagnosis analysis of MIMIC-IV. [WIP]
1. Environment Setup & Initial Model Loading
2. Data Cleaning & Filtering
3. Data Splitting & Target Isolation
4. Prompt Formating & Input Structure
5. Model Inference & Structured Output Generation
6. Label Matching & Alignment
7. Analysis & Performance Evaluation

## Sources and Developers [WIP]

Comparative study developed by John E. Griffith (jeg5275)

MedGemma 1.5 27b developed by Google's HAI-DEF (Health AI Developer Foundations)

MIMIC-IV developed by the researchers at the Massachusetts Institute of Technology, and gathered by the Beth Israel Deaconess Medical Center

Label Matching developed by [WIP]




## Environment Setup and Initial Model Loading

`pip install -U transformers accelerate torch`
or
`pip install -U transformers`
or
`pip install -U transformers accelerate torch huggingface_hub`

In [ ]:
import torch
from huggingface_hub import login
from transformers import pipeline

# >>> CONFIG PARAMETERS <<<

# Model details
MODEL_ID = "google/medgemma-27b-text-it"
HF_TOKEN = None # ... set to "hf_xxx..." or keep as none to use interactive login.

# Model hardware and precision settings
TORCH_DTYPE = torch.bfloat16
DEVICE_MAP = "cuda" # cuda, cpu, auto

# Model parameters
MAX_NEW_TOKENS = 256
DO_SAMPLE = True
TEMPERATURE = 0.6
TOP_P = 0.9

# -----> Prompt and Message input <-----
SYSTEM_PROMPT = "You are a helpful medical assistant."
USER_QUERY = "In 4 sentences, how can you differentiate acute versus chronic radiation poisoning in practice?" # sample, but will change to direct input or file

# checks if needs authentication to login, otherwise continue
def authenticate_huggingface(token=None):
  if token:
    login(token=token)
  else:
    print("Authenticating with HuggingFace...")
    login()
    print("Authentication Successful.")


# initializes MG through pipeline
def initialize__model(model_id, token=None):
  medical_pipeline = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=TORCH_DTYPE,
    device_map=DEVICE_MAP,
  )
  # returns initialized
  return medical_pipeline

# function to actual generate response, given initalized model (pipe), system and user input
def generate_response(pipe, system_content, user_content):

  # inital inputs
  messages = [
      {"role": "system", "content": system_content},
      {"role": "user", "content": user_content}
  ]

  # system output
  outputs = pipe(
      messages,
      max_new_tokens=MAX_NEW_TOKENS,
      do_sample=DO_SAMPLE,
      temperature=TEMPERATURE,
      top_p=TOP_P,
  )

  # return outout
  return outputs

In [ ]:
# initial
if __name__ == "__main__":

  # login or continue with huggingface
  authenticate_huggingface(token=HF_TOKEN)

  # loads model
  print("Loading MedGemma-27b-text-it model...")
  pipe = initialize__model(MODEL_ID, token=HF_TOKEN)
  print("Model loaded.")

  # inputs and generates response
  print("Generating Response...")
  response = generate_response(pipe, SYSTEM_PROMPT, USER_QUERY)
  print("Response generated.")

  # outputs response
  final_output = response[0]['generated_text'][-1]["content"]
  print("\n ----- MedGemma Output -----")
  print(final_output)

Authenticating with HuggingFace...


Authentication Successful.
Loading MedGemma-27b-text-it model...


config.json:   0%|          | 0.00/931 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/67.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 11 files:   0%|          | 0/11 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/808 [00:00<?, ?it/s]

ValueError: Could not load model google/medgemma-27b-text-it with any of the following classes: (<class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>, <class 'transformers.models.gemma3.modeling_gemma3.Gemma3ForCausalLM'>). See the original errors:

while loading with AutoModelForCausalLM, an error is thrown:
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/transformers/pipelines/base.py", line 241, in load_model
    model = model_class.from_pretrained(model, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/transformers/models/auto/auto_factory.py", line 406, in from_pretrained
    return model_class.from_pretrained(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        pretrained_model_name_or_path, *model_args, config=config, **hub_kwargs, **kwargs
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py", li